# Static `env` schema analysis with AST

This notebook contains a self-contained implementation of `analyze_env_schema(function)` and validates it against all eight requested rules.

The analyzer is a path-sensitive symbolic interpreter. It never runs the analyzed function. It tracks the original `env` tree, independently tracked deep copies, active name aliases, overwrite state, usage state, direct root arguments, and logical execution paths.

### Deliberate semantics

- `branch = env["a"]` creates an alias and schema nodes but does not mark them used.
- Passing a tracked **tree root** directly to any function records a `FunctionCall` and does not mark that root used. This includes `copy.deepcopy(env)`, because it is also a direct root argument.
- Passing a branch, such as `consume(env["a"])`, is a normal use rather than a `FunctionCall`.
- Branch outputs are compared after remaining common statements. A conditional access can therefore converge when another path performs the same access later.
- Literal subscription keys are required. Dynamic keys raise `DynamicKeyError`.
- `DeepDiff` is used automatically when installed. A built-in structural diff keeps the notebook self-contained when it is absent.

## Imports

In [1]:
import jmaps as jm
from jmaps import (
    normalized_function_ast,
    EnvTree,
    EnvNode,
    analyze_env_schema,
    FunctionCall,
    jmap,
    PathOptions,
)
from jmaps.journey.schema import print_analysis, CopyOrigin, DynamicSchemaError, DynamicKeyError, logger
import json

## Validation fixtures

In [2]:
import linecache
import textwrap
import types


def module_from_source(source: str, name: str, extra_globals=None):
    """Compile source under a linecache-backed filename for inspect.getsource."""
    source = textwrap.dedent(source).lstrip("\n")
    if not source.endswith("\n"):
        source += "\n"
    filename = f"<{name}>"
    lines = source.splitlines(keepends=True)
    linecache.cache[filename] = (len(source), None, lines, filename)
    module = types.ModuleType(name)
    module.__file__ = filename
    if extra_globals:
        module.__dict__.update(extra_globals)
    exec(compile(source, filename, "exec"), module.__dict__)
    return module


def node(tree: EnvTree, *path):
    """Get an already-created node without changing the analyzed tree."""
    return tree.get_node(path, create=False)


cases = module_from_source(
    r'''
    import copy
    from copy import deepcopy

    def rule1(env):
        pending = env["branch"]["leaf"]
        env["branch"]
        env["other"]["leaf"]

    def rule2(env):
        known = env["known"]["leaf"]
        full = copy.deepcopy(env)
        source = env["source"]
        partial = deepcopy(source)
        full["full_leaf"]
        partial["partial_leaf"]

    def rule3(env):
        env2 = env
        env3 = env2
        branch = env3["a"]
        deep = branch["b"]
        deep["leaf"]
        deep = object()
        deep["ignored"]
        env2 = copy.deepcopy(env)
        env2["copy_leaf"]
        env3 = None
        env3["ignored_root"]

    def rule4(env):
        known = env["branch"]["leaf"]["known"]
        env["branch"]["leaf"] = 0
        env["branch"]["leaf"]["late"]
        env["branch"]["sibling"]
        env["branch"]
        env["target"] = env["rhs"]

    def rule5(env):
        untouched = env["not_overwritten"]
        env["before"]["x"] = 1
        first(env)
        env["after"]["y"] = 2
        clone = copy.deepcopy(env)
        clone["copy_only"] = 3
        service.second(payload=clone)
        consume(env["before"])

    def rule6(env):
        alias_only = env["alias_only"]
        env["overwrite_only"] = 1
        send(env)
        env["used"]["leaf"]

    def consistent_if_elif(env, mode):
        if mode == 0:
            env["same"]
            forward(env)
        elif mode == 1:
            env["same"]
            forward(env)
        else:
            env["same"]
            forward(env)

    def divergent_if(env, flag):
        if flag:
            env["left"]
        else:
            env["right"]

    def converges_after_if(env, flag):
        if flag:
            env["common"]
        env["common"]

    def consistent_match(env, value):
        match value:
            case 1:
                env["same_match"]
            case _:
                env["same_match"]

    def divergent_match(env, value):
        match value:
            case 1:
                env["match_left"]
            case _:
                env["match_right"]

    def nonexhaustive_match(env, value):
        match value:
            case 1:
                env["only_if_one"]

    def consistent_try(env):
        try:
            env["same_try"]
            forward(env)
        except ValueError:
            env["same_try"]
            forward(env)

    def divergent_try(env):
        try:
            env["try_side"]
        except ValueError:
            env["except_side"]

    def dynamic_key(env, key):
        env[key]

    def assignment_evaluation_order(env):
        alias = env["value"]
        alias = alias + 1
    ''',
    "env_schema_validation_cases",
)

validation_results = []


def passed(label: str):
    validation_results.append(label)
    print("✓", label)


## Rule 1 — tree nodes and `used` propagation

In [3]:
# Rule 1: parent/child tree structure, literal keys, and recursive use marking.
trees, calls = analyze_env_schema(cases.rule1)
root = trees[0].root

assert root.children["branch"].used
assert root.children["branch"].children["leaf"].used
assert not root.children["other"].used
assert root.children["other"].children["leaf"].used
assert calls == []

passed("Rule 1 — tree structure and used flags")
print_analysis(trees, calls)


✓ Rule 1 — tree structure and used flags
{
  "trees": [
    {
      "tree_id": 0,
      "copied_from": null,
      "reduced_to_overwrites": false,
      "root": {
        "key": {
          "type": "str",
          "repr": "'<root:env>'"
        },
        "used": false,
        "overwritten": false,
        "children": [
          {
            "key": {
              "type": "str",
              "repr": "'branch'"
            },
            "used": true,
            "overwritten": false,
            "children": [
              {
                "key": {
                  "type": "str",
                  "repr": "'leaf'"
                },
                "used": true,
                "overwritten": false,
                "children": []
              }
            ]
          },
          {
            "key": {
              "type": "str",
              "repr": "'other'"
            },
            "used": false,
            "overwritten": false,
            "children": [
              

## Rules 2–3 — deep copies and active aliases

In [4]:
# Rule 2: every deepcopy becomes an independent EnvTree with origin metadata.
trees, calls = analyze_env_schema(cases.rule2)
assert len(trees) == 3
assert trees[1].copied_from == CopyOrigin(0, ())
assert trees[2].copied_from == CopyOrigin(0, ("source",))
assert node(trees[1], "known", "leaf").used is False
assert node(trees[1], "full_leaf").used is True
assert node(trees[2], "partial_leaf").used is True
assert [call.function_name for call in calls] == ["copy.deepcopy"]
passed("Rule 2 — independently tracked deep copies")

# Rule 3: aliases can be chained to arbitrary depth and are removed on rebinding.
trees, calls = analyze_env_schema(cases.rule3)
assert node(trees[0], "a", "b", "leaf").used
assert "ignored" not in node(trees[0], "a", "b").children
assert "ignored_root" not in trees[0].root.children
assert len(trees) == 2
assert node(trees[1], "copy_leaf").used
passed("Rule 3 — arbitrary-depth aliases and alias override")


✓ Rule 2 — independently tracked deep copies
✓ Rule 3 — arbitrary-depth aliases and alias override


In [5]:
for tree in trees:
    print(tree.get_used_leaves())

{('a', 'b', 'leaf')}
{('a', 'b', 'leaf'), ('copy_leaf',)}


## Rule 4 — overwrite state

In [6]:
# Rule 4: overwrites propagate to existing descendants; new descendants inherit
# overwritten=True; reads of overwritten nodes do not alter used.
trees, calls = analyze_env_schema(cases.rule4)
branch = node(trees[0], "branch")
leaf = node(trees[0], "branch", "leaf")

assert branch.used and node(trees[0], "branch", "sibling").used
assert leaf.overwritten and not leaf.used
assert node(trees[0], "branch", "leaf", "known").overwritten
assert not node(trees[0], "branch", "leaf", "known").used
assert node(trees[0], "branch", "leaf", "late").overwritten
assert not node(trees[0], "branch", "leaf", "late").used
assert node(trees[0], "target").overwritten
assert not node(trees[0], "target").used
assert node(trees[0], "rhs").used

passed("Rule 4 — overwrite propagation and read suppression")


✓ Rule 4 — overwrite propagation and read suppression


## Rule 5 — `FunctionCall` snapshots

In [7]:
# Rule 5: direct root arguments create independent reduced snapshots.
trees, calls = analyze_env_schema(cases.rule5)
print(calls)
assert [call.function_name for call in calls] == [
    "first",
    "copy.deepcopy",
    "service.second",
]

first_snapshot = calls[0].overwritten_tree
copy_snapshot = calls[1].overwritten_tree
service_snapshot = calls[2].overwritten_tree

# The first snapshot cannot change when later overwrites occur.
assert node(first_snapshot, "before", "x").overwritten
assert not node(first_snapshot, "before").used
assert "after" not in first_snapshot.root.children
assert "not_overwritten" not in first_snapshot.root.children

# The deepcopy call sees both original overwrites at its own call time.
assert node(copy_snapshot, "before", "x").overwritten
assert node(copy_snapshot, "after", "y").overwritten

# The copied root later contains inherited overwrites plus its own overwrite.
assert node(service_snapshot, "before", "x").overwritten
assert node(service_snapshot, "after", "y").overwritten
assert node(service_snapshot, "copy_only").overwritten

# Passing env["before"] is a branch use, not a root FunctionCall.
assert node(trees[0], "before").used
assert not node(trees[0], "before", "x").used

passed("Rule 5 — root-call snapshots and reduced overwrite trees")


[FunctionCall(function_name='first', overwritten_tree={
  "tree_id": 0,
  "copied_from": null,
  "reduced_to_overwrites": true,
  "root": {
    "key": {
      "type": "str",
      "repr": "'<root:env>'"
    },
    "used": false,
    "overwritten": false,
    "children": [
      {
        "key": {
          "type": "str",
          "repr": "'before'"
        },
        "used": false,
        "overwritten": false,
        "children": [
          {
            "key": {
              "type": "str",
              "repr": "'x'"
            },
            "used": false,
            "overwritten": true,
            "children": []
          }
        ]
      }
    ]
  }
}), FunctionCall(function_name='copy.deepcopy', overwritten_tree={
  "tree_id": 0,
  "copied_from": null,
  "reduced_to_overwrites": true,
  "root": {
    "key": {
      "type": "str",
      "repr": "'<root:env>'"
    },
    "used": false,
    "overwritten": false,
    "children": [
      {
        "key": {
          "type": "st

In [8]:
for tree in trees:
    print(tree)
    print(tree.get_used_leaves())

{
  "tree_id": 0,
  "copied_from": null,
  "reduced_to_overwrites": false,
  "root": {
    "key": {
      "type": "str",
      "repr": "'<root:env>'"
    },
    "used": false,
    "overwritten": false,
    "children": [
      {
        "key": {
          "type": "str",
          "repr": "'after'"
        },
        "used": false,
        "overwritten": false,
        "children": [
          {
            "key": {
              "type": "str",
              "repr": "'y'"
            },
            "used": false,
            "overwritten": true,
            "children": []
          }
        ]
      },
      {
        "key": {
          "type": "str",
          "repr": "'before'"
        },
        "used": true,
        "overwritten": false,
        "children": [
          {
            "key": {
              "type": "str",
              "repr": "'x'"
            },
            "used": false,
            "overwritten": true,
            "children": []
          }
        ]
      },
      

## Rules 6–7 — usage exclusions and outputs

In [9]:
# Rule 6: alias creation, overwrite targets, and direct root passing are not uses.
trees, calls = analyze_env_schema(cases.rule6)
assert not node(trees[0], "alias_only").used
assert node(trees[0], "overwrite_only").overwritten
assert not node(trees[0], "overwrite_only").used
assert not trees[0].root.used
assert not node(trees[0], "used").used
assert node(trees[0], "used", "leaf").used
assert [call.function_name for call in calls] == ["send"]
passed("Rule 6 — usage exceptions")

# Rule 7: exact requested output container types.
assert isinstance(trees, list)
assert isinstance(calls, list)
assert all(isinstance(tree, EnvTree) for tree in trees)
assert all(isinstance(call, FunctionCall) for call in calls)
passed("Rule 7 — output types")


✓ Rule 6 — usage exceptions
✓ Rule 7 — output types


## Rule 8 — control-flow consistency

In [10]:
# Rule 8: consistent paths pass.
analyze_env_schema(cases.consistent_if_elif)
analyze_env_schema(cases.converges_after_if)
analyze_env_schema(cases.consistent_match)
analyze_env_schema(cases.consistent_try)

# Divergent paths raise and expose branch-specific structural differences.
expected_failures = [
    cases.divergent_if,
    cases.divergent_match,
    cases.nonexhaustive_match,
    cases.divergent_try,
]


captured = {}
logger_was_disabled = logger.disabled
logger.disabled = True  # Avoid four expected ERROR records in notebook output.
try:
    for function in expected_failures:
        try:
            analyze_env_schema(function)
        except DynamicSchemaError as exc:
            assert exc.branch_differences
            assert exc.branch_differences[0]["diff"]
            captured[function.__name__] = exc.branch_differences[0]
        else:
            raise AssertionError(
                f"Expected DynamicSchemaError for {function.__name__}"
            )
finally:
    logger.disabled = logger_was_disabled

passed("Rule 8 — if/elif, match-case, and try-except consistency")
print("Example divergent-if diagnostic:")
print(json.dumps(captured["divergent_if"], indent=2, default=repr))


✓ Rule 8 — if/elif, match-case, and try-except consistency
Example divergent-if diagnostic:
{
  "baseline_paths": [
    "entry -> if@66:body"
  ],
  "compared_paths": [
    "entry -> if@66:else"
  ],
  "diff": {
    "values_changed": {
      "root['trees'][0]['root']['children'][0]['key']['repr']": {
        "new_value": "'right'",
        "old_value": "'left'"
      }
    }
  }
}


## Additional static-analysis checks

In [11]:
# Additional safety: dynamic keys are rejected rather than guessed.
try:
    analyze_env_schema(cases.dynamic_key)
except DynamicKeyError:
    pass
else:
    raise AssertionError("Expected DynamicKeyError")
passed("Additional — dynamic-key rejection")

# Assignment RHS is analyzed before rebinding the target alias.
trees, _ = analyze_env_schema(cases.assignment_evaluation_order)
assert node(trees[0], "value").used
passed("Additional — Python assignment evaluation order")

print(f"\n{len(validation_results)} validation groups passed.")


✓ Additional — dynamic-key rejection
✓ Additional — Python assignment evaluation order

10 validation groups passed.


## Practical boundaries

The implementation is intentionally conservative where Python cannot be resolved statically:

- Keys must be literals. Supporting dynamic keys would require a symbolic-key representation or runtime tracing.
- Deep-copy recognition is name based. Add aliases such as `"cp.deepcopy"` through `deepcopy_names=`.
- A `try` handler is modeled as a logical alternative beginning at try entry. Precise exception-prefix analysis would require exception-effect modeling.
- Loops are additionally supported as zero iterations versus one-or-more iterations. Loop-dependent copies or calls are therefore reported as dynamic.
- Reflection, mutations hidden inside arbitrary functions, and aliases stored inside containers require interprocedural analysis or runtime tracing.

# Compare two functions for equivalence (ignoring formatting)

In [12]:
def functions_structurally_equal(func1, func2):
    """
    Determine whether two functions have the same normalized AST.
    """
    return normalized_function_ast(func1) == normalized_function_ast(func2)


# ---------------------------------------------------------------------
# Example
# ---------------------------------------------------------------------

def function_a(x):
    """Square x and add one."""
    # This is a comment.
    y = x ** 2
    return y + 1


def completely_different_name(x):

    y=x**2

    return y+1


def function_c(x):
    y = x ** 3
    return y + 1


print(functions_structurally_equal(function_a, completely_different_name))
# True

print(functions_structurally_equal(function_a, function_c))
# False

print(normalized_function_ast(function_a))

True
False
Module(body=[FunctionDef(name='<function>', args=arguments(args=[arg(arg='x')]), body=[Assign(targets=[Name(id='y', ctx=Store())], value=BinOp(left=Name(id='x', ctx=Load()), op=Pow(), right=Constant(value=2))), Return(value=BinOp(left=Name(id='y', ctx=Load()), op=Add(), right=Constant(value=1)))])])


In [13]:
import hashlib
signature_a = hashlib.sha256(
    normalized_function_ast(function_a).encode()
).hexdigest()
signature_b = hashlib.sha256(
    normalized_function_ast(completely_different_name).encode()
).hexdigest()
print(signature_a)
print(signature_b)
assert signature_a == signature_b

34f06ab23763a5a0e841c55765b4e06177d624da86aaf0087fefd04b422c7ff0
34f06ab23763a5a0e841c55765b4e06177d624da86aaf0087fefd04b422c7ff0


# Path Result

# Function Decorator

In [14]:
@jmap(nondeterministic=False)
def my_path(env, path_options, partial_result):
    """Test comments for MY PATH!!!"""
    print("Hello world")
    b = env["b"]
    b["1"]
    a = env["a"]
    two = env["b"]["2"]

@jmap
def my_super_path(env, path_options, partial_result):
    """Test comments for MY SUPER PATH!!!"""
    print("Hello world")
    my_path(env, path_options)
    env["b"]["2"]

@jmap
def my_super_overwritten_path(env, path_options, partial_result):
    """Test comments for MY SUPER OVERWRITTEN PATH!!!"""
    print("Hello world")
    env["list"][0]
    b = env["b"]
    b["1"] = 1
    my_path(env, path_options)
    s = "" + env["b"]["2"].__name__

env = {
    "a" : "a",
    "b" : {
        "1" : {0: 0},
        "2" : 2
    },
    "list" : [1, 2, 3]
}
path_options = PathOptions(disable_saving_and_loading=True)
# The IDE knows the original call signature.
result = my_path(env, path_options)

try:
    @jmap
    def my_path_error(env, partial_result):
        pass
except TypeError as e:
    print(f"Caught expected TypeError: {e}")

# The IDE also knows that this is a JPath:
print(f"Schema: {my_path.get_used_leaves()}")

# And it knows JPath attributes:
original_function = my_path.func

# Runtime introspection also works.
# (env: int, partial_result: str, path_options: bool) -> int
print(f"Path Registry: {jm.journey.path.jpath_registry}")
print("######################################################################")
print(f"Super Usage: {my_super_path.get_used_leaves()}")
print(f"Super Function Call Overwrites: {[f"{call.function_name}: {call.overwritten_tree.get_overwritten_leaves()}" for call in my_super_path.function_calls]}")
print(f"Super Env Schema: {my_super_path.evaluate_schema(env)}")
print("######################################################################")
print(f"Super Overwritten Usage: {my_super_overwritten_path.get_used_leaves()}")
print(f"Super Function Call Overwrites: {[f"{call.function_name}: {call.overwritten_tree.get_overwritten_leaves()}" for call in my_super_overwritten_path.function_calls]}")
print(f"Super Overwritten Env Schema: {my_super_overwritten_path.evaluate_schema(env)}")


Hello world
Caught expected TypeError: @jmap requires arguments ('env', 'path_options', 'partial_result'), in that order. my_path_error has arguments ('env', 'partial_result').
Schema: {('b', '1')}
Path Registry: {'__main__.my_path': <jmaps.journey.path.JPath object at 0x770d38ff6660>, '__main__.my_super_path': <jmaps.journey.path.JPath object at 0x770d38eb4a50>, '__main__.my_super_overwritten_path': <jmaps.journey.path.JPath object at 0x770d38eb4910>}
######################################################################
Super Usage: {('b', '2'), ('b', '1')}
Super Function Call Overwrites: ['my_path: set()']
Super Env Schema: ({'format': 'jmaps.environment-schema.v2', 'root': {'kind': 'container', 'adapter': 'builtins.dict', 'metadata': {}, 'children': {'b': {'selector': {'type': 'str', 'value': 'b'}, 'schema': {'kind': 'container', 'adapter': 'builtins.dict', 'metadata': {}, 'children': {'1': {'selector': {'type': 'str', 'value': '1'}, 'schema': {'kind': 'container', 'adapter': 'buil